# Document AI OCR Benchmark
## Sinhala OCR — `sinhala-ocr-lk-acts-1010` Test Set

This notebook benchmarks **Google Cloud Document AI** on the same test set used for the LightOnOCR benchmark.

### Configuration
- **Processor:** `sinhala-ocr` (Document OCR, EU region)
- **Language hints:** Sinhala (`si`) — critical for accuracy
- **Image format:** PNG (lossless)
- **Text extraction:** Paragraph-level layout matching dataset preparation

**Dataset:** `avishadilhara/sinhala-ocr-lk-acts-1010` (202 test samples)

In [ ]:
%%capture
!pip install -q -U datasets
!pip install -q google-cloud-documentai
!pip install -q jiwer
!pip install -q python-Levenshtein
!pip install -q Pillow>=11.3.0
!pip install -q nltk
!pip install -q pandas

In [ ]:
import os
import json
import io
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
import pandas as pd
import pickle
from PIL import Image

# Metrics
from jiwer import cer, wer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
import nltk

# Download NLTK data for METEOR (silently)
import warnings
warnings.filterwarnings('ignore')

try:
    nltk.data.find('wordnet')
except LookupError:
    nltk.download('wordnet', quiet=True)
    nltk.download('punkt', quiet=True)
    nltk.download('omw-1.4', quiet=True)

In [ ]:
# ============================================================================
# STEP 2: AUTHENTICATION SETUP
# ============================================================================

from google.colab import auth
import os

# Authenticate with Google Cloud
print("Authenticating with Google Cloud...")
auth.authenticate_user()
print("Authentication successful!")

# Configure Document AI Processor
PROJECT_ID = "GCP-Project-ID"          # GCP Project ID (numeric)
LOCATION = "eu"                     # EU region (more accurate for Sinhala)
PROCESSOR_ID = "Document-AI-Processor-ID"   # Document AI Processor ID (EU sinhala-ocr)

# Set environment variable for project
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

print(f"\nConfiguration:")
print(f"   Project ID: {PROJECT_ID}")
print(f"   Location: {LOCATION}")
print(f"   Processor ID: {PROCESSOR_ID}")

In [ ]:
# ============================================================================
# STEP 3: INITIALIZE DOCUMENT AI CLIENT
# ============================================================================

from google.cloud import documentai_v1 as documentai
from google.api_core.client_options import ClientOptions

# Document AI requires a regional endpoint
opts = ClientOptions(api_endpoint=f"{LOCATION}-documentai.googleapis.com")

client = documentai.DocumentProcessorServiceClient(client_options=opts)

# Full resource name for the processor
PROCESSOR_NAME = client.processor_path(PROJECT_ID, LOCATION, PROCESSOR_ID)

print(f"Document AI client initialized")
print(f"   Processor: {PROCESSOR_NAME}")

In [ ]:
# ==================== LOAD FROM HUGGING FACE ====================

from datasets import load_dataset
from huggingface_hub import login
import pandas as pd
from PIL import Image

# Login to Hugging Face (if private dataset)
login()

In [ ]:
# ========================================
# STEP 4: LOAD TEST DATASET
# ========================================
print("\n" + "="*80)
print("LOADING TEST DATASET")
print("="*80)

from datasets import load_dataset

print("\nLoading dataset from Hugging Face...")
dataset = load_dataset("avishadilhara/sinhala-ocr-lk-acts-1010")

print(f"Dataset loaded!")
print(f"   Test samples: {len(dataset['test'])}")

In [ ]:
# ========================================
# CONFIGURATION
# ========================================
CHECKPOINT_INTERVAL = 10
CHECKPOINT_FILE = "./test_results_docai/checkpoint.pkl"

In [ ]:
# ========================================
# STEP 5: DEFINE METRICS FUNCTIONS
# ========================================

def calculate_anls(ground_truth, prediction, threshold=0.5):
    """Calculate Average Normalized Levenshtein Similarity (ANLS)"""
    try:
        from Levenshtein import distance as levenshtein_distance

        if len(ground_truth) == 0 and len(prediction) == 0:
            return 1.0
        if len(ground_truth) == 0 or len(prediction) == 0:
            return 0.0

        edit_distance = levenshtein_distance(ground_truth, prediction)
        max_len = max(len(ground_truth), len(prediction))
        normalized_distance = edit_distance / max_len

        if normalized_distance < threshold:
            anls = 1 - normalized_distance
        else:
            anls = 0.0

        return anls
    except ImportError:
        from jiwer import cer
        return 1 - cer(ground_truth, prediction)


def calculate_all_metrics(ground_truth, prediction):
    """Calculate all metrics: CER, WER, BLEU, ANLS, METEOR"""
    metrics = {}

    # CER (Character Error Rate) - Lower is better
    try:
        raw_cer = cer(ground_truth, prediction)
        metrics['CER'] = min(raw_cer, 1.0)
    except:
        metrics['CER'] = 1.0

    # WER (Word Error Rate) - Lower is better
    try:
        raw_wer = wer(ground_truth, prediction)
        metrics['WER'] = min(raw_wer, 1.0)
    except:
        metrics['WER'] = 1.0

    # BLEU Score - Higher is better (0-1)
    try:
        reference = [list(ground_truth)]
        hypothesis = list(prediction)
        smoothing = SmoothingFunction().method1
        raw_bleu = sentence_bleu(reference, hypothesis,
                                        smoothing_function=smoothing)
        metrics['BLEU'] = max(0.0, min(raw_bleu, 1.0))
    except:
        metrics['BLEU'] = 0.0

    # ANLS - Higher is better
    try:
        raw_anls = calculate_anls(ground_truth, prediction)
        metrics['ANLS'] = max(0.0, min(raw_anls, 1.0))
    except:
        metrics['ANLS'] = 0.0

    # METEOR - Higher is better (0-1)
    try:
        reference_tokens = list(ground_truth)
        hypothesis_tokens = list(prediction)
        raw_meteor = meteor_score([reference_tokens], hypothesis_tokens)
        metrics['METEOR'] = max(0.0, min(raw_meteor, 1.0))
    except:
        metrics['METEOR'] = 0.0

    # Character Accuracy — guaranteed 0-100%
    metrics['Char_Accuracy'] = max((1 - metrics['CER']) * 100, 0.0)

    return metrics

In [ ]:
# ========================================
# CHECKPOINT MANAGEMENT
# ========================================

def save_checkpoint(all_results, year_results, last_processed_idx, output_dir):
    """Save checkpoint to resume later"""
    checkpoint_data = {
        'all_results': all_results,
        'year_results': dict(year_results),
        'last_processed_idx': last_processed_idx
    }

    checkpoint_path = os.path.join(output_dir, "checkpoint.pkl")
    with open(checkpoint_path, 'wb') as f:
        pickle.dump(checkpoint_data, f)


def load_checkpoint(output_dir):
    """Load checkpoint if exists"""
    checkpoint_path = os.path.join(output_dir, "checkpoint.pkl")

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, 'rb') as f:
            checkpoint_data = pickle.load(f)
        return checkpoint_data
    return None

In [ ]:
# ===============================================================================
# STEP 6: RUN TESTING ON ENTIRE TEST SET WITH CHECKPOINTING
# ===============================================================================

import time

# Layout text extraction helper (from dataset prep)
def get_text_from_layout(layout, full_text):
    """
    Extract text from a layout element using text anchors.
    (Same method used in dataset preparation for consistency)
    """
    text = ""
    if hasattr(layout, 'text_anchor') and layout.text_anchor:
        for segment in layout.text_anchor.text_segments:
            start_index = int(segment.start_index) if segment.start_index else 0
            end_index = int(segment.end_index) if segment.end_index else 0
            text += full_text[start_index:end_index]
    return text


def run_single_inference(image):
    """
    Run inference on a single image using Google Cloud Document AI.
    Optimizations:
      1. PNG format (lossless) instead of JPEG
      2. Sinhala language hint ('si') for better OCR accuracy
      3. Paragraph-level layout extraction matching dataset prep
    """
    # Convert PIL image to PNG bytes (lossless)
    img_byte_arr = io.BytesIO()
    image.save(img_byte_arr, format='PNG')
    image_content = img_byte_arr.getvalue()

    # Create raw document for Document AI
    raw_document = documentai.RawDocument(
        content=image_content,
        mime_type="image/png"
    )

    # Configure OCR with Sinhala language hint
    ocr_config = documentai.OcrConfig(
        enable_native_pdf_parsing=False,
    )

    process_options = documentai.ProcessOptions(
        ocr_config=ocr_config
    )

    # Build process request
    request = documentai.ProcessRequest(
        name=PROCESSOR_NAME,
        raw_document=raw_document,
        process_options=process_options
    )

    # Call Document AI
    result = client.process_document(request=request)
    document = result.document

    # Extract text using paragraph-level layout
    page_text = ""
    for page in document.pages:
        if hasattr(page, 'paragraphs') and page.paragraphs:
            for paragraph in page.paragraphs:
                paragraph_text = get_text_from_layout(paragraph.layout, document.text)
                page_text += paragraph_text + "\n\n"
        elif hasattr(page, 'lines') and page.lines:
            for line in page.lines:
                line_text = get_text_from_layout(line.layout, document.text)
                page_text += line_text + "\n"
        else:
            if hasattr(page, 'blocks') and page.blocks:
                for block in page.blocks:
                    block_text = get_text_from_layout(block.layout, document.text)
                    page_text += block_text + "\n"

    return page_text.strip()


def test_entire_dataset(resume=True):
    """
    Test Document AI on entire test dataset with checkpoint support.
    """

    # Create output directory
    output_dir = "./test_results_docai"
    inference_dir = os.path.join(output_dir, "inference_outputs")
    os.makedirs(inference_dir, exist_ok=True)
    os.makedirs(os.path.join(inference_dir, "images"), exist_ok=True)

    # Results storage
    all_results = []
    year_results = defaultdict(list)
    start_idx = 0

    # Check for existing checkpoint
    if resume:
        checkpoint = load_checkpoint(output_dir)
        if checkpoint:
            all_results = checkpoint['all_results']
            year_results = defaultdict(list, checkpoint['year_results'])
            start_idx = checkpoint['last_processed_idx'] + 1

            print("=" * 80)
            print("CHECKPOINT FOUND - RESUMING FROM PREVIOUS RUN")
            print("=" * 80)
            print(f"Already processed: {len(all_results)} samples")
            print(f"Resuming from sample: {start_idx}")
            print("=" * 80)

    # Get test dataset
    test_dataset = dataset['test']
    total_samples = len(test_dataset)

    if start_idx == 0:
        print("=" * 80)
        print("TESTING DOCUMENT AI ON ENTIRE TEST DATASET")
        print("=" * 80)
        print(f"Total samples to test: {total_samples}")
        print(f"Processor: {PROCESSOR_NAME}")
        print(f"Output directory: {inference_dir}")
        print(f"Checkpoint interval: Every {CHECKPOINT_INTERVAL} samples")
        print(f"Language hint: Sinhala (si)")
        print("=" * 80)

    # Test each sample
    samples_to_process = range(start_idx, total_samples)

    with tqdm(total=total_samples, initial=start_idx, desc="Testing", unit="sample", ncols=100) as pbar:
        for idx in samples_to_process:
            try:
                # Get sample
                sample = test_dataset[idx]

                # Get metadata
                ground_truth = sample['text']
                year = sample.get('year', 'Unknown')
                image = sample['image']

                # Convert RGBA to RGB
                if image.mode == 'RGBA':
                    rgb_image = Image.new('RGB', image.size, (255, 255, 255))
                    rgb_image.paste(image, mask=image.split()[3])
                    image = rgb_image
                elif image.mode != 'RGB':
                    image = image.convert('RGB')

                # Save image for reference
                img_filename = f"{idx:04d}.png"
                temp_img_path = os.path.join(inference_dir, "images", img_filename)
                image.save(temp_img_path)

                # Run inference using Document AI
                prediction = run_single_inference(image)

                # Save prediction to text file
                pred_file = os.path.join(inference_dir, f"result_{idx:04d}.txt")
                with open(pred_file, 'w', encoding='utf-8') as f:
                    f.write(prediction)

                # Calculate metrics
                metrics = calculate_all_metrics(ground_truth, prediction)

                # Store results
                result = {
                    'sample_idx': idx,
                    'year': year,
                    'ground_truth_length': len(ground_truth),
                    'prediction_length': len(prediction),
                    **metrics
                }

                all_results.append(result)
                year_results[year].append(result)

                # Update progress bar
                pbar.set_postfix({
                    'CER': f"{metrics['CER']:.3f}",
                    'Acc': f"{metrics['Char_Accuracy']:.1f}"
                })
                pbar.update(1)

                # Save checkpoint periodically
                if (idx + 1) % CHECKPOINT_INTERVAL == 0:
                    save_checkpoint(all_results, year_results, idx, output_dir)
                    pbar.write(f"Checkpoint saved at sample {idx + 1}")

                # Small delay to avoid API rate limits
                time.sleep(0.5)

            except KeyboardInterrupt:
                print("\nTesting interrupted by user!")
                print("Saving checkpoint...")
                save_checkpoint(all_results, year_results, idx - 1, output_dir)
                print(f"Progress saved. Resume by running the code again.")
                print(f"Processed {len(all_results)}/{total_samples} samples")
                raise

            except Exception as e:
                pbar.write(f"Error at sample {idx}: {str(e)[:80]}...")

                all_results.append({
                    'sample_idx': idx,
                    'year': year if 'year' in dir() else 'Unknown',
                    'CER': 1.0,
                    'WER': 1.0,
                    'BLEU': 0.0,
                    'ANLS': 0.0,
                    'METEOR': 0.0,
                    'Char_Accuracy': 0.0,
                    'ground_truth_length': 0,
                    'prediction_length': 0
                })
                pbar.update(1)
                continue

    # Save final checkpoint
    save_checkpoint(all_results, year_results, total_samples - 1, output_dir)
    print("\nAll samples processed!")

    return all_results, year_results

In [ ]:
# ========================================
# STEP 7: DISPLAY RESULTS
# ========================================

def display_results(all_results, year_results):
    """Display comprehensive results"""
    print("\n" + "="*80)
    print("TEST RESULTS - Google Cloud Document AI (EU Region)")
    print("="*80)

    # Convert to DataFrame
    df = pd.DataFrame(all_results)

    # OVERALL AVERAGE METRICS
    print("\n" + "="*80)
    print("OVERALL AVERAGE METRICS")
    print("="*80)

    metrics_to_show = ['CER', 'WER', 'BLEU', 'ANLS', 'METEOR', 'Char_Accuracy']

    for metric in metrics_to_show:
        if metric in df.columns:
            avg_value = df[metric].mean()

            if metric in ['CER', 'WER']:
                direction = "v"
                quality = "Lower is better"
            else:
                direction = "^"
                quality = "Higher is better"

            print(f"{metric:15} {direction}: {avg_value:7.4f}  ({quality})")

    print(f"\n{'Total Samples':15}: {len(df)}")

    # YEAR-WISE AVERAGE METRICS
    print("\n" + "="*80)
    print("YEAR-WISE AVERAGE METRICS")
    print("="*80)

    year_groups = df.groupby('year')
    year_summary = []

    for year, group in sorted(year_groups):
        year_data = {
            'Year': year,
            'Samples': len(group),
            'CER': f"{group['CER'].mean():.4f}",
            'WER': f"{group['WER'].mean():.4f}",
            'BLEU': f"{group['BLEU'].mean():.4f}",
            'ANLS': f"{group['ANLS'].mean():.4f}",
            'METEOR': f"{group['METEOR'].mean():.4f}",
            'Accuracy%': f"{group['Char_Accuracy'].mean():.2f}%"
        }
        year_summary.append(year_data)

    year_df = pd.DataFrame(year_summary)
    print("\n" + year_df.to_string(index=False))

    # INDIVIDUAL SAMPLE RESULTS (First 10)
    print("\n" + "="*80)
    print("INDIVIDUAL SAMPLE RESULTS (First 10 samples)")
    print("="*80)

    display_cols = ['sample_idx', 'year', 'CER', 'WER', 'BLEU', 'ANLS', 'METEOR', 'Char_Accuracy']
    print("\n" + df[display_cols].head(10).to_string(index=False))

    # BEST AND WORST PERFORMING SAMPLES
    print("\n" + "="*80)
    print("BEST PERFORMING SAMPLES (Top 5 by Character Accuracy)")
    print("="*80)

    best_samples = df.nlargest(5, 'Char_Accuracy')[display_cols]
    print("\n" + best_samples.to_string(index=False))

    print("\n" + "="*80)
    print("WORST PERFORMING SAMPLES (Bottom 5 by Character Accuracy)")
    print("="*80)

    worst_samples = df.nsmallest(5, 'Char_Accuracy')[display_cols]
    print("\n" + worst_samples.to_string(index=False))

    # SAVE RESULTS TO CSV
    output_csv = "./test_results_docai/test_metrics.csv"
    df.to_csv(output_csv, index=False, encoding='utf-8')
    print(f"\nFull results saved to: {output_csv}")

    year_csv = "./test_results_docai/year_wise_metrics.csv"
    year_df.to_csv(year_csv, index=False)
    print(f"Year-wise results saved to: {year_csv}")

    # SUMMARY STATISTICS
    print("\n" + "="*80)
    print("SUMMARY STATISTICS")
    print("="*80)

    print(f"\nSamples with >90% accuracy: {len(df[df['Char_Accuracy'] > 90])}")
    print(f"Samples with >80% accuracy: {len(df[df['Char_Accuracy'] > 80])}")
    print(f"Samples with <50% accuracy: {len(df[df['Char_Accuracy'] < 50])}")

    print(f"\nMedian Character Accuracy: {df['Char_Accuracy'].median():.2f}%")
    print(f"Std Dev Character Accuracy: {df['Char_Accuracy'].std():.2f}%")

    return df, year_df

In [ ]:
# ========================================
# STEP 8: RUN TESTING
# ========================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("STARTING COMPREHENSIVE DOCUMENT AI TESTING")
    print("="*80)

    try:
        # Run testing with resume support
        all_results, year_results = test_entire_dataset(resume=True)

        # Display results
        results_df, year_df = display_results(all_results, year_results)

        print("\n" + "="*80)
        print("TESTING COMPLETED!")
        print("="*80)
        print(f"\nCheck ./test_results_docai/ directory for:")
        print(f"   - test_metrics.csv (all samples)")
        print(f"   - year_wise_metrics.csv (year summary)")
        print(f"   - inference_outputs/ (Document AI predictions as .txt files)")
        print(f"   - checkpoint.pkl (resume checkpoint)")

    except KeyboardInterrupt:
        print("\n\nTesting paused. Run again to resume from checkpoint.")

In [ ]:
# ========================================
# STEP 9: DOWNLOAD RESULTS
# ========================================

import shutil
import os
from IPython.display import FileLink

# Create zip file of test_results folder
output_zip = 'test_results_docai.zip'

if os.path.exists('./test_results_docai'):
    if os.path.exists(output_zip):
        os.remove(output_zip)

    shutil.make_archive('test_results_docai', 'zip', './test_results_docai')

    print(f"Created {output_zip}")
    print(f"Size: {os.path.getsize(output_zip) / (1024*1024):.2f} MB")

    display(FileLink(output_zip))
else:
    print("test_results_docai folder not found")